# LeetCode 1065 — Index Pairs of a String bằng Python

**Link đề:** https://leetcode.com/problems/index-pairs-of-a-string/
**Độ khó:** Easy · **Chủ đề:** String, Trie, Sorting, Array

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
class Solution:
    def indexPairs(self, text: str, words: list[str]) -> list[list[int]]:
        ...
```

## Mục tiêu học
1. Hiểu yêu cầu: liệt kê mọi cặp chỉ số `[i, j]` mà đoạn `text[i..j]` (tính cả hai đầu) là một từ trong `words`, theo đúng **thứ tự sắp xếp**.
2. Nắm slicing `text[i:j+1]` (chỉ số "đóng" vs "nửa mở"), `set` để tra cứu nhanh, sắp xếp list các cặp.
3. Viết 3 lời giải: duyệt mọi đoạn con + `set` → tìm từng từ bằng `str.find`/`startswith` → **Trie** (cây tiền tố).
4. Trace Trie từng bước, chứng minh tính đúng, so sánh Time/Space.
5. Tự làm các biến thể (đếm số lần xuất hiện, in đậm từ khóa…).

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — tìm mọi vị trí các từ trong một văn bản

Cho chuỗi `text` và mảng chuỗi `words`. Trả về mọi cặp chỉ số `[i, j]` sao cho đoạn con `text[i], text[i+1], ..., text[j]` **thuộc** `words`.

Kết quả phải **sắp xếp**: theo `i` tăng dần; nếu `i` bằng nhau thì theo `j` tăng dần.

**Ví dụ 1:** `text = "thestoryofleetcodeandme"`, `words = ["story","fleet","leetcode"]`
→ `[[3,7],[9,13],[10,17]]`

**Ví dụ 2:** `text = "ababa"`, `words = ["aba","ab"]`
→ `[[0,1],[0,2],[2,3],[2,4]]` — các lần xuất hiện **được phép chồng nhau** (`"aba"` ở 0..2 và 2..4).

**Constraints (theo hiểu biết):**
- `1 <= text.length <= 100`
- `1 <= words.length <= 20`, `1 <= words[i].length <= 50`
- `text`, `words[i]` gồm chữ cái thường; các từ trong `words` **phân biệt**.

In [1]:
text = "thestoryofleetcodeandme"
for i, j in [[3, 7], [9, 13], [10, 17]]:
    print(f"[{i}, {j}] -> text[{i}:{j + 1}] = {text[i:j + 1]!r}")

[3, 7] -> text[3:8] = 'story'
[9, 13] -> text[9:14] = 'fleet'
[10, 17] -> text[10:18] = 'leetcode'


## 2. Nền tảng Python cần biết

### 2.1. Chỉ số "đóng" `[i, j]` và slicing nửa mở `text[i:j+1]`

Đề dùng cặp **đóng** `[i, j]` (lấy cả `j`). Slicing Python `text[a:b]` là **nửa mở** (không lấy `b`). Vì vậy:

```text
đoạn [i, j]  <=>  text[i : j + 1]       và   j = i + len(word) - 1
```

In [2]:
text = "ababa"
word = "aba"
i = 2
j = i + len(word) - 1
print("i =", i, "j =", j, "| text[i:j+1] =", text[i:j + 1])
print("text[i:j] (thiếu 1 ký tự) =", text[i:j])

i = 2 j = 4 | text[i:j+1] = aba
text[i:j] (thiếu 1 ký tự) = ab


### 2.2. `set` — tra cứu "có thuộc tập không" nhanh

`set` lưu các phần tử **không trùng**, kiểm tra `x in s` trung bình `O(1)` (với chuỗi: cần băm chuỗi, tốn `O(độ dài chuỗi)`). Kiểm tra `x in list` thì phải dò từng phần tử.

In [3]:
words = ["story", "fleet", "leetcode"]
word_set = set(words)
print(word_set)
print("story" in word_set, "stor" in word_set)

{'story', 'fleet', 'leetcode'}
True False


### 2.3. Sắp xếp list các cặp

`sorted` / `.sort()` so sánh list theo **thứ tự từ điển** (lexicographic): so phần tử đầu, bằng nhau thì so phần tử thứ hai… Đúng thứ tự đề yêu cầu!

In [4]:
pairs = [[2, 4], [0, 2], [2, 3], [0, 1]]
print(sorted(pairs))      # [[0, 1], [0, 2], [2, 3], [2, 4]]
print([0, 2] < [2, 3], [2, 3] < [2, 4], [0, 9] < [1, 0])

[[0, 1], [0, 2], [2, 3], [2, 4]]
True True True


### 2.4. `str.startswith(prefix, start)` và `str.find(sub, start)`

- `text.startswith(w, i)` → `True` nếu `text` **tại vị trí `i`** bắt đầu bằng `w`.
- `text.find(w, start)` → chỉ số **đầu tiên `>= start`** nơi `w` xuất hiện, hoặc `-1` nếu không có.

In [5]:
text = "ababa"
print(text.startswith("aba", 0), text.startswith("aba", 1), text.startswith("aba", 2))
print(text.find("aba"), text.find("aba", 1), text.find("aba", 3))

True False True
0 2 -1


## 3. Cách 1 — Duyệt mọi đoạn con, kiểm tra bằng `set` (baseline)

Với mọi `i` từ `0..n-1` và mọi `j` từ `i..n-1`: nếu `text[i:j+1] in word_set` thì thêm `[i, j]`.

Duyệt `i` tăng dần, trong đó `j` tăng dần ⇒ kết quả **tự động đã sắp** — không cần `sort`!

In [6]:
def index_pairs_brute(text: str, words: list[str]) -> list[list[int]]:
    word_set = set(words)
    n = len(text)
    res = []
    for i in range(n):
        for j in range(i, n):
            if text[i:j + 1] in word_set:
                res.append([i, j])
    return res

print(index_pairs_brute("thestoryofleetcodeandme", ["story", "fleet", "leetcode"]))
print(index_pairs_brute("ababa", ["aba", "ab"]))

[[3, 7], [9, 13], [10, 17]]
[[0, 1], [0, 2], [2, 3], [2, 4]]


### Cải tiến nhỏ: giới hạn độ dài

Một từ dài tối đa `L = max(len(w))`, nên `j` chỉ cần chạy tới `i + L - 1`. Tránh tạo những đoạn con quá dài vô ích.

In [7]:
def index_pairs_brute_limited(text: str, words: list[str]) -> list[list[int]]:
    word_set = set(words)
    L = max(len(w) for w in words)
    n = len(text)
    res = []
    for i in range(n):
        for j in range(i, min(n, i + L)):
            if text[i:j + 1] in word_set:
                res.append([i, j])
    return res

print(index_pairs_brute_limited("ababa", ["aba", "ab"]))

[[0, 1], [0, 2], [2, 3], [2, 4]]


## 4. Cách 2 — Duyệt theo từng từ, tìm mọi lần xuất hiện

Đảo góc nhìn: với **mỗi từ** `w`, tìm mọi vị trí `i` mà `text` bắt đầu bằng `w` tại `i`. Ghi `[i, i + len(w) - 1]`.

Vì duyệt theo từ, kết quả **chưa sắp** → phải `sort` ở cuối.

Dùng `find` lặp: sau khi tìm thấy tại `i`, tìm tiếp từ `i + 1` (không phải `i + len(w)`) để **không bỏ lỡ lần xuất hiện chồng nhau**.

In [8]:
def index_pairs_find(text: str, words: list[str]) -> list[list[int]]:
    res = []
    for w in words:
        i = text.find(w)
        while i != -1:
            res.append([i, i + len(w) - 1])
            i = text.find(w, i + 1)       # +1 để bắt được lần xuất hiện chồng nhau
    res.sort()
    return res

print(index_pairs_find("thestoryofleetcodeandme", ["story", "fleet", "leetcode"]))
print(index_pairs_find("ababa", ["aba", "ab"]))

[[3, 7], [9, 13], [10, 17]]
[[0, 1], [0, 2], [2, 3], [2, 4]]


## 5. Nút thắt & insight: dùng chung tiền tố — Trie

Hai cách trên kiểm tra **từng từ độc lập** hoặc **từng đoạn con độc lập**. Nhưng các phép so sánh bị lặp:

- Từ `"ab"` và `"aba"` có chung tiền tố `"ab"`. Khi đứng tại `i` mà `text[i:]` không bắt đầu bằng `"a"`, ta biết ngay **cả hai** đều không khớp.
- Với cách 1, `text[i:j+1]` được tạo mới (copy) cho mỗi `j`, dù đoạn `text[i:j]` vừa xét xong.

**Insight:** gom các từ vào **Trie** (cây tiền tố). Đứng tại `i`, đi xuống Trie theo `text[i], text[i+1], ...`:
- Mỗi bước chỉ đi theo **1 cạnh** (một ký tự).
- Nút nào đánh dấu "kết thúc từ" → ghi `[i, j]`.
- Không có cạnh cho ký tự tiếp → **dừng ngay**, vì không từ nào có tiền tố đó.

### 5.1. Trie là gì?

Trie là cây mà **mỗi cạnh là một ký tự**; đường đi từ gốc xuống một nút chính là một **tiền tố**. Nút có cờ `end = True` nếu tiền tố đó là một từ hoàn chỉnh.

Với `words = ["ab", "aba"]`:

```text
(gốc)
  └─ a
      └─ b   (end: "ab")
          └─ a   (end: "aba")
```

Trong Python, cách gọn nhất: **mỗi nút là một `dict`** `{ký tự: nút con}`; dùng key đặc biệt `"$"` để đánh dấu kết thúc từ.

In [9]:
def build_trie(words: list[str]) -> dict:
    root = {}
    for w in words:
        node = root
        for ch in w:
            node = node.setdefault(ch, {})   # có rồi thì lấy, chưa có thì tạo {}
        node["$"] = True                     # đánh dấu kết thúc từ
    return root

trie = build_trie(["ab", "aba"])
print(trie)

{'a': {'b': {'$': True, 'a': {'$': True}}}}


### 5.2. `dict.setdefault(key, default)`

- Nếu `key` **đã có** → trả về giá trị đang có.
- Nếu **chưa có** → gán `d[key] = default` rồi trả về `default`.

Nhờ vậy `node = node.setdefault(ch, {})` vừa tạo nút con (nếu cần) vừa đi xuống nút đó trong **một dòng**.

In [10]:
d = {}
x = d.setdefault("a", {})
print(d, x is d["a"])        # {'a': {}} True
y = d.setdefault("a", {"khác": 1})
print(d, y)                  # không đổi: key 'a' đã có

{'a': {}} True
{'a': {}} {}


## 6. Cách 3 (lời giải chính) — Trie, ghép từng đoạn

**Đoạn 1 — dựng Trie** từ `words` (như trên).

**Đoạn 2 — với mỗi điểm bắt đầu `i`**: `node = root`.

**Đoạn 3 — đi xuống:** `for j in range(i, n)`:
- Nếu `text[j]` không có trong `node` → `break`.
- `node = node[text[j]]`; nếu `"$" in node` → `res.append([i, j])`.

Duyệt `i` tăng, `j` tăng ⇒ kết quả đã sắp, **không cần `sort`**.

In [11]:
def index_pairs_trie(text: str, words: list[str]) -> list[list[int]]:
    root = build_trie(words)                 # Đoạn 1
    n = len(text)
    res = []
    for i in range(n):                       # Đoạn 2
        node = root
        for j in range(i, n):                # Đoạn 3
            ch = text[j]
            if ch not in node:
                break                        # không từ nào có tiền tố này
            node = node[ch]
            if "$" in node:
                res.append([i, j])
    return res

print(index_pairs_trie("thestoryofleetcodeandme", ["story", "fleet", "leetcode"]))
print(index_pairs_trie("ababa", ["aba", "ab"]))

[[3, 7], [9, 13], [10, 17]]
[[0, 1], [0, 2], [2, 3], [2, 4]]


## 7. Trace Trie với `text = "ababa"`, `words = ["aba", "ab"]`

| `i` | Đi qua ký tự (`j`) | Nút `end`? | Ghi nhận | Dừng vì |
|:---:|---|---|---|---|
| 0 | a(0) → b(1) → a(2) → b(3)? | `ab` ở j=1, `aba` ở j=2 | `[0,1]`, `[0,2]` | sau `aba` không có cạnh `b` |
| 1 | b(1)? | — | — | gốc không có cạnh `b` |
| 2 | a(2) → b(3) → a(4) | `ab` ở j=3, `aba` ở j=4 | `[2,3]`, `[2,4]` | hết chuỗi |
| 3 | b(3)? | — | — | gốc không có cạnh `b` |
| 4 | a(4) → hết chuỗi | — | — | hết chuỗi |

In [12]:
def trace_index_pairs_trie(text: str, words: list[str]) -> list[list[int]]:
    root = build_trie(words)
    res = []
    for i in range(len(text)):
        node, path = root, ""
        for j in range(i, len(text)):
            ch = text[j]
            if ch not in node:
                print(f"i={i}: sau {path!r} không có cạnh {ch!r} -> dừng")
                break
            node = node[ch]
            path += ch
            if "$" in node:
                res.append([i, j])
                print(f"i={i}: đi tới {path!r} (j={j}) là TỪ -> ghi [{i}, {j}]")
        else:
            print(f"i={i}: đi tới {path!r}, hết chuỗi")
    print("Kết quả:", res)
    return res

trace_index_pairs_trie("ababa", ["aba", "ab"])

i=0: đi tới 'ab' (j=1) là TỪ -> ghi [0, 1]
i=0: đi tới 'aba' (j=2) là TỪ -> ghi [0, 2]
i=0: sau 'aba' không có cạnh 'b' -> dừng
i=1: sau '' không có cạnh 'b' -> dừng
i=2: đi tới 'ab' (j=3) là TỪ -> ghi [2, 3]
i=2: đi tới 'aba' (j=4) là TỪ -> ghi [2, 4]
i=2: đi tới 'aba', hết chuỗi
i=3: sau '' không có cạnh 'b' -> dừng
i=4: đi tới 'a', hết chuỗi
Kết quả: [[0, 1], [0, 2], [2, 3], [2, 4]]


[[0, 1], [0, 2], [2, 3], [2, 4]]

### Cú pháp `for ... else`

Khối `else` của vòng `for` chạy **khi vòng lặp kết thúc bình thường** (không gặp `break`). Ở hàm trace, `else` báo "hết chuỗi" — tức ta không dừng vì thiếu cạnh.

In [13]:
for x in [1, 2, 3]:
    if x == 5:
        break
else:
    print("không gặp break -> chạy else")

for x in [1, 5, 3]:
    if x == 5:
        print("gặp 5 -> break")
        break
else:
    print("dòng này KHÔNG in")

không gặp break -> chạy else
gặp 5 -> break


## 8. Phiên bản Trie bằng class (dễ đọc, hay gặp trong phỏng vấn)

Thay `dict` lồng nhau bằng class `TrieNode` có `children` (dict) và `is_end` (bool).

In [14]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_end = False

def index_pairs_trie_class(text: str, words: list[str]) -> list[list[int]]:
    root = TrieNode()
    for w in words:
        node = root
        for ch in w:
            if ch not in node.children:
                node.children[ch] = TrieNode()
            node = node.children[ch]
        node.is_end = True

    res = []
    for i in range(len(text)):
        node = root
        for j in range(i, len(text)):
            node = node.children.get(text[j])   # get trả None nếu không có
            if node is None:
                break
            if node.is_end:
                res.append([i, j])
    return res

print(index_pairs_trie_class("ababa", ["aba", "ab"]))

[[0, 1], [0, 2], [2, 3], [2, 4]]


## 9. Code nộp LeetCode

### Bản Trie (lời giải chính)

In [15]:
class Solution:
    def indexPairs(self, text: str, words: list[str]) -> list[list[int]]:
        root = {}
        for w in words:
            node = root
            for ch in w:
                node = node.setdefault(ch, {})
            node["$"] = True

        res = []
        n = len(text)
        for i in range(n):
            node = root
            for j in range(i, n):
                if text[j] not in node:
                    break
                node = node[text[j]]
                if "$" in node:
                    res.append([i, j])
        return res

sol = Solution()
assert sol.indexPairs("thestoryofleetcodeandme", ["story", "fleet", "leetcode"]) == [[3, 7], [9, 13], [10, 17]]
assert sol.indexPairs("ababa", ["aba", "ab"]) == [[0, 1], [0, 2], [2, 3], [2, 4]]
print("Solution (Trie) PASS")

Solution (Trie) PASS


### Bản `startswith` (ngắn, đủ nhanh với constraints nhỏ)

In [16]:
class SolutionStartswith:
    def indexPairs(self, text: str, words: list[str]) -> list[list[int]]:
        res = []
        for i in range(len(text)):
            for w in words:
                if text.startswith(w, i):
                    res.append([i, i + len(w) - 1])
        res.sort()
        return res

assert SolutionStartswith().indexPairs("ababa", ["aba", "ab"]) == [[0, 1], [0, 2], [2, 3], [2, 4]]
print("SolutionStartswith PASS")

SolutionStartswith PASS


Lưu ý: ở bản `startswith`, cùng một `i` các từ được duyệt theo thứ tự trong `words` (có thể `"aba"` trước `"ab"`), nên vẫn cần `sort` để `j` tăng dần.

## 10. Vì sao Trie đúng?

**Mệnh đề:** tại điểm bắt đầu `i`, sau khi xử lý ký tự `text[j]` mà chưa `break`, `node` là nút Trie ứng với tiền tố `text[i:j+1]`.

- Dựng Trie: mỗi từ `w` tạo đúng đường đi `w[0], w[1], ...` từ gốc, nút cuối có `"$"`. Vậy một chuỗi `t` là từ trong `words` ⇔ đi theo `t` từ gốc tới được một nút có `"$"`.
- Khi `text[j]` không có trong `node`: không từ nào có tiền tố `text[i:j+1]`, nên không có `[i, j']` nào với `j' >= j` → `break` an toàn (**không bỏ sót**).
- Mỗi lần ghi `[i, j]` là khi `text[i:j+1]` đúng là một từ (**không ghi nhầm**).
- Thứ tự: `i` tăng ở vòng ngoài, `j` tăng ở vòng trong ⇒ đúng thứ tự sắp yêu cầu; mỗi cặp `[i, j]` được xét đúng một lần ⇒ không trùng.

## 11. Độ phức tạp

Ký hiệu: `n = len(text)`, `m = len(words)`, `L` = độ dài từ dài nhất, `S` = tổng độ dài các từ, `k` = số cặp kết quả.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Mọi đoạn con + `set` | `O(n² · n)` (tạo/băm đoạn dài tới `n`); bản giới hạn `L`: `O(n · L²)` | `O(S)` |
| 2. `find` từng từ + `sort` | `O(m · n · L)` worst + `O(k log k)` | `O(k)` |
| 3. Trie | dựng `O(S)` + tìm `O(n · L)` | `O(S)` cho Trie |
| `startswith` | `O(n · m · L)` + `O(k log k)` | `O(k)` |

Với `n <= 100`, mọi cách đều nhanh; Trie là cách **mở rộng tốt** khi có nhiều từ chung tiền tố hoặc `text` dài.

## 12. Bộ test

Nhóm test:
1. Hai ví dụ đề.
2. Không có kết quả.
3. Từ là tiền tố của từ khác, chồng nhau: `"aaaa"`, `["a","aa"]`.
4. Từ dài hơn `text`.
5. Cả `text` là một từ.
6. Đối chiếu ngẫu nhiên mọi cách.

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:22} PASS -> {actual}")

tests = [
    ("đề 1", "thestoryofleetcodeandme", ["story", "fleet", "leetcode"], [[3, 7], [9, 13], [10, 17]]),
    ("đề 2", "ababa", ["aba", "ab"], [[0, 1], [0, 2], [2, 3], [2, 4]]),
    ("không có", "abc", ["d", "xyz"], []),
    ("chồng nhau a/aa", "aaaa", ["a", "aa"],
     [[0, 0], [0, 1], [1, 1], [1, 2], [2, 2], [2, 3], [3, 3]]),
    ("từ dài hơn text", "ab", ["abc"], []),
    ("cả text là từ", "leet", ["leet", "e"], [[0, 3], [1, 1], [2, 2]]),
]
for name, text, words, exp in tests:
    check(name, index_pairs_trie(text, words), exp)

đề 1                   PASS -> [[3, 7], [9, 13], [10, 17]]
đề 2                   PASS -> [[0, 1], [0, 2], [2, 3], [2, 4]]
không có               PASS -> []
chồng nhau a/aa        PASS -> [[0, 0], [0, 1], [1, 1], [1, 2], [2, 2], [2, 3], [3, 3]]
từ dài hơn text        PASS -> []
cả text là từ          PASS -> [[0, 3], [1, 1], [2, 2]]


In [18]:
import random
random.seed(1065)
for _ in range(1500):
    text = "".join(random.choice("ab") for _ in range(random.randint(1, 15)))
    pool = {"".join(random.choice("ab") for _ in range(random.randint(1, 4))) for _ in range(6)}
    words = list(pool)                           # từ phân biệt
    results = [
        index_pairs_brute(text, words),
        index_pairs_brute_limited(text, words),
        index_pairs_find(text, words),
        index_pairs_trie(text, words),
        index_pairs_trie_class(text, words),
        SolutionStartswith().indexPairs(text, words),
    ]
    assert all(r == results[0] for r in results), (text, words, results)
print("1500 test ngẫu nhiên: 6 cách khớp nhau")

1500 test ngẫu nhiên: 6 cách khớp nhau


## 13. Lỗi người mới hay gặp

### Lỗi 1: Nhầm chỉ số đóng/mở — ghi `[i, i + len(w)]`

Đề dùng `j` **tính cả**: `j = i + len(w) - 1`. Ghi `i + len(w)` lệch 1.

### Lỗi 2: `find` tiếp từ `i + len(w)` → bỏ lỡ lần xuất hiện chồng nhau

In [19]:
def wrong_skip_overlap(text, words):
    res = []
    for w in words:
        i = text.find(w)
        while i != -1:
            res.append([i, i + len(w) - 1])
            i = text.find(w, i + len(w))     # nhảy qua cả từ -> mất "aba" ở vị trí 2
    return sorted(res)

print("sai :", wrong_skip_overlap("ababa", ["aba", "ab"]))
print("đúng:", index_pairs_find("ababa", ["aba", "ab"]))

sai : [[0, 1], [0, 2], [2, 3]]
đúng: [[0, 1], [0, 2], [2, 3], [2, 4]]


### Lỗi 3: Dừng ở từ khớp đầu tiên tại mỗi `i` (`break` sau khi ghi)

Với Trie, gặp nút kết thúc **không có nghĩa** là dừng: từ dài hơn có thể khớp tiếp (`"ab"` rồi `"aba"`).

### Lỗi 4: Quên sắp xếp khi duyệt theo từ

In [20]:
def wrong_no_sort(text, words):
    res = []
    for w in words:
        i = text.find(w)
        while i != -1:
            res.append([i, i + len(w) - 1])
            i = text.find(w, i + 1)
    return res                                # thiếu sort

def wrong_break_after_match(text, words):
    root = build_trie(words)
    res = []
    for i in range(len(text)):
        node = root
        for j in range(i, len(text)):
            if text[j] not in node:
                break
            node = node[text[j]]
            if "$" in node:
                res.append([i, j])
                break                         # sai: dừng quá sớm
    return res

print("thiếu sort :", wrong_no_sort("ababa", ["aba", "ab"]))
print("break sớm  :", wrong_break_after_match("ababa", ["aba", "ab"]))
print("đúng       :", index_pairs_trie("ababa", ["aba", "ab"]))

thiếu sort : [[0, 2], [2, 4], [0, 1], [2, 3]]
break sớm  : [[0, 1], [2, 3]]
đúng       : [[0, 1], [0, 2], [2, 3], [2, 4]]


## 14. Mẹo nhận dạng pattern

- **Nhiều từ** cần tìm trong **một văn bản**, hoặc hỏi về **tiền tố** → nghĩ tới **Trie**.
- Đề yêu cầu kết quả "sắp theo (i, j)" → nếu duyệt `i` rồi `j` tăng dần thì có thứ tự sẵn; nếu duyệt theo từ thì nhớ `sort`.
- Constraints nhỏ (`n <= 100`) → `startswith`/`set` đủ qua; Trie là để học kỹ thuật tổng quát.

| Bài liên quan | Liên hệ |
|---|---|
| 208. Implement Trie | xây Trie: insert / search / startsWith |
| 616 / 758. Add Bold Tag / Bold Words in String | đánh dấu vùng khớp rồi gộp khoảng |
| 1032. Stream of Characters | Trie trên từ đảo ngược |
| 139. Word Break | Trie hoặc `set` + DP |

## 15. Biến thể

### Biến thể A — Đánh dấu ký tự thuộc ít nhất một từ (ý tưởng bài in đậm từ khóa)

In [21]:
def bold_mask(text: str, words: list[str]) -> str:
    mask = [False] * len(text)
    for i, j in index_pairs_trie(text, words):
        for k in range(i, j + 1):
            mask[k] = True
    return "".join(ch.upper() if m else ch for ch, m in zip(text, mask))

print(bold_mask("thestoryofleetcodeandme", ["story", "fleet", "leetcode"]))

theSTORYoFLEETCODEandme


### Biến thể B — Đếm số lần mỗi từ xuất hiện (cho phép chồng nhau)

In [22]:
from collections import Counter

def count_occurrences(text: str, words: list[str]) -> dict:
    cnt = Counter()
    for i, j in index_pairs_trie(text, words):
        cnt[text[i:j + 1]] += 1
    return {w: cnt[w] for w in words}

print(count_occurrences("ababa", ["aba", "ab", "zz"]))

{'aba': 2, 'ab': 2, 'zz': 0}


## 16. Bài tập tự luyện (tăng dần)

**Bài 1 (tính tay):** `text = "aabaa"`, `words = ["a", "aa", "baa"]`. Liệt kê kết quả đúng thứ tự.

**Bài 2 (điền chỗ trống):** hoàn thiện `ex_build_trie(words)` theo TODO.

**Bài 3 (viết lại):** `ex_index_pairs(text, words)` dùng `ex_build_trie`.

**Bài 4 (biến thể):** `ex_longest_at_each(text, words)` — với mỗi `i` có ít nhất một từ bắt đầu tại `i`, chỉ giữ cặp `[i, j]` có `j` **lớn nhất**.

**Bài 5 (mini challenge):** `ex_merge_ranges(pairs)` — gộp các đoạn `[i, j]` đã sắp mà **chồng nhau hoặc kề nhau** (`j + 1 >= i'`) thành đoạn lớn. Đây là bước then chốt của bài "in đậm từ khóa".

In [23]:
def ex_build_trie(words: list[str]) -> dict:
    root = {}
    for w in words:
        node = root
        # TODO 1: đi qua từng ký tự, tạo nút con nếu chưa có
        # TODO 2: đánh dấu kết thúc từ bằng key "$"
        pass
    return root

# assert ex_build_trie(["ab"]) == {"a": {"b": {"$": True}}}

In [24]:
def ex_index_pairs(text: str, words: list[str]) -> list[list[int]]:
    # TODO: dựng trie; với mỗi i đi xuống trie theo text[j]; gặp "$" thì ghi [i, j]
    pass

# assert ex_index_pairs("aabaa", ["a", "aa", "baa"]) == [[0, 0], [0, 1], [1, 1], [2, 4], [3, 3], [3, 4], [4, 4]]

In [25]:
def ex_longest_at_each(text: str, words: list[str]) -> list[list[int]]:
    # TODO: dùng kết quả index pairs; với cùng i, giữ j lớn nhất
    pass

# assert ex_longest_at_each("ababa", ["aba", "ab"]) == [[0, 2], [2, 4]]

In [26]:
def ex_merge_ranges(pairs: list[list[int]]) -> list[list[int]]:
    # TODO: duyệt pairs đã sắp; nếu đoạn mới bắt đầu <= cuối đoạn trước + 1 thì nới cuối
    pass

# assert ex_merge_ranges([[0, 1], [0, 2], [2, 3], [2, 4]]) == [[0, 4]]
# assert ex_merge_ranges([[3, 7], [9, 13], [10, 17]]) == [[3, 7], [9, 17]]

## 17. Gợi ý 3 mức

**Bài 1:** (1) Với từng `i`, liệt kê từ bắt đầu tại `i`. (2) `i=0`: `a`, `aa`; `i=1`: `a`; `i=2`: `baa`; `i=3`: `a`, `aa`; `i=4`: `a`. (3) Đổi sang cặp `[i, i+len-1]`.

**Bài 2:** (1) Giống `build_trie`. (2) `node = node.setdefault(ch, {})`. (3) Sau vòng ký tự: `node["$"] = True`.

**Bài 3:** (1) Như mục 6. (2) `if text[j] not in node: break`. (3) `if "$" in node: res.append([i, j])`.

**Bài 4:** (1) Kết quả đã sắp theo `(i, j)` → cặp cuối cùng của mỗi `i` có `j` lớn nhất. (2) Dùng dict `best[i] = j` ghi đè dần. (3) `[[i, best[i]] for i in sorted(best)]`.

**Bài 5:** (1) Giữ đoạn đang gộp `[s, e]`. (2) Nếu `i <= e + 1` thì `e = max(e, j)`. (3) Ngược lại đẩy đoạn cũ vào kết quả, mở đoạn mới.

## 18. Đáp án tham khảo + template + checklist

Tự làm trước rồi mới xem. Bài 1: `[[0,0],[0,1],[1,1],[2,4],[3,3],[3,4],[4,4]]`.

In [27]:
def ans_build_trie(words):
    root = {}
    for w in words:
        node = root
        for ch in w:
            node = node.setdefault(ch, {})
        node["$"] = True
    return root

def ans_index_pairs(text, words):
    root = ans_build_trie(words)
    res = []
    for i in range(len(text)):
        node = root
        for j in range(i, len(text)):
            if text[j] not in node:
                break
            node = node[text[j]]
            if "$" in node:
                res.append([i, j])
    return res

def ans_longest_at_each(text, words):
    best = {}
    for i, j in ans_index_pairs(text, words):
        best[i] = j                     # cặp sau cùng của mỗi i có j lớn nhất
    return [[i, best[i]] for i in sorted(best)]

def ans_merge_ranges(pairs):
    merged = []
    for i, j in pairs:
        if merged and i <= merged[-1][1] + 1:
            merged[-1][1] = max(merged[-1][1], j)
        else:
            merged.append([i, j])
    return merged

assert ans_build_trie(["ab"]) == {"a": {"b": {"$": True}}}
assert ans_index_pairs("aabaa", ["a", "aa", "baa"]) == [[0, 0], [0, 1], [1, 1], [2, 4], [3, 3], [3, 4], [4, 4]]
assert ans_longest_at_each("ababa", ["aba", "ab"]) == [[0, 2], [2, 4]]
assert ans_merge_ranges([[0, 1], [0, 2], [2, 3], [2, 4]]) == [[0, 4]]
assert ans_merge_ranges([[3, 7], [9, 13], [10, 17]]) == [[3, 7], [9, 17]]
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


### Template nhớ nhanh

```python
class Solution:
    def indexPairs(self, text: str, words: list[str]) -> list[list[int]]:
        root = {}
        for w in words:                          # 1) dựng Trie
            node = root
            for ch in w:
                node = node.setdefault(ch, {})
            node["$"] = True
        res = []
        for i in range(len(text)):               # 2) mỗi điểm bắt đầu
            node = root
            for j in range(i, len(text)):        # 3) đi xuống Trie
                if text[j] not in node:
                    break
                node = node[text[j]]
                if "$" in node:
                    res.append([i, j])
        return res                               # đã sắp theo (i, j)
```

### Checklist trước khi submit
- [ ] `j` là chỉ số **đóng**: `j = i + len(w) - 1`, slicing là `text[i:j+1]`.
- [ ] Không bỏ lỡ lần xuất hiện **chồng nhau**.
- [ ] Gặp từ khớp **không** dừng — từ dài hơn có thể khớp tiếp.
- [ ] Kết quả sắp theo `(i, j)` (tự nhiên với Trie; nhớ `sort` nếu duyệt theo từ).
- [ ] Test `"ababa"`, `["aba","ab"]` → `[[0,1],[0,2],[2,3],[2,4]]`.